In [ ]:
#%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

Looking in indexes: https://download.pytorch.org/whl/cu118
INFO: pip is looking at multiple versions of torchaudio to determine which version is compatible with other requirements. This could take a while.
     ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
     ---------------------------------------- 4.0/4.0 MB 34.3 MB/s eta 0:00:00
     ---------------------------------------- 0.0/2.7 GB ? eta -:--:--
     ---------------------------------------- 0.0/2.7 GB 78.1 MB/s eta 0:00:35
      --------------------------------------- 0.0/2.7 GB 88.5 MB/s eta 0:00:31
      --------------------------------------- 0.0/2.7 GB 81.0 MB/s eta 0:00:33
      --------------------------------------- 0.0/2.7 GB 54.2 MB/s eta 0:00:50
      --------------------------------------- 0.1/2.7 GB 59.8 MB/s eta 0:00:45
     - -------------------------------------- 0.1/2.7 GB 62.1 MB/s eta 0:00:43
     - -------------------------------------- 0.1/2.7 GB 50.1 MB/s eta 0:00:53
     - -------------

  You can safely remove it manually.


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm import tqdm
import pandas as pd

In [2]:
# Load dataset
df = pd.read_csv(r"H:\THESIS\DATASET\filtered_data_len32.csv")
protein_seqs = df["seq"].tolist()
ss8_labels = df["sst8"].tolist()

In [3]:
# Apply TF-IDF (Trigram only)
def apply_tfidf_trigrams(sequences):
    vectorizer = TfidfVectorizer(analyzer="char", ngram_range=(3, 3))
    tfidf_matrix = vectorizer.fit_transform(sequences)
    return tfidf_matrix, vectorizer

tfidf_matrix, vectorizer = apply_tfidf_trigrams(protein_seqs)

# Prepare labels (padded to max length)
def prepare_labels(labels):
    unique_labels = sorted(set("".join(labels)))
    label_map = {char: idx for idx, char in enumerate(unique_labels)}
    max_length = max(len(seq) for seq in labels)

    y = []
    for seq in labels:
        padded_seq = [label_map[char] for char in seq]
        padded_seq += [0] * (max_length - len(seq))  # Pad with 0 (assuming 0 is padding class)
        y.append(padded_seq)

    return torch.tensor(y, dtype=torch.long), label_map

In [ ]:
y, label_map = prepare_labels(ss8_labels)
max_length = y.shape[1]  # Maximum sequence length

# Create sparse graph (validate edge_index later)
def create_sparse_graph(sequences, tfidf_matrix, threshold=0.5):
    edge_index = []
    edge_weight = []

    tfidf_sparse = tfidf_matrix.tocsr()
    for i in tqdm(range(tfidf_sparse.shape[0]), desc="Building graph"):
        row_i = tfidf_sparse.getrow(i).toarray().flatten()
        for j in range(i + 1, tfidf_sparse.shape[0]):
            row_j = tfidf_sparse.getrow(j).toarray().flatten()
            sim = 1 - torch.cosine_similarity(
                torch.tensor(row_i), torch.tensor(row_j), dim=0
            ).item()
            if sim >= threshold:
                edge_index.append([i, j])
                edge_index.append([j, i])  # Undirected graph
                edge_weight.append(sim)
                edge_weight.append(sim)

    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    edge_weight = torch.tensor(edge_weight, dtype=torch.float)
    return edge_index, edge_weight

edge_index, edge_weight = create_sparse_graph(protein_seqs, tfidf_matrix)

Building graph:  79%|███████▉  | 12317/15559 [2:27:56<11:20,  4.76it/s]     

In [2]:
# Validate and fix edge_index
def validate_and_fix_edge_index(edge_index, num_nodes):
    mask = (edge_index[0] < num_nodes) & (edge_index[1] < num_nodes)
    return edge_index[:, mask]

edge_index = validate_and_fix_edge_index(edge_index, tfidf_matrix.shape[0])

# Prepare PyTorch Geometric Data object
x = torch.tensor(tfidf_matrix.toarray(), dtype=torch.float)
gnn_data = Data(x=x, edge_index=edge_index, y=y)

In [ ]:
# Define the GCN Model
class GCN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, max_length):
        super(GCN, self).__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.max_length = max_length

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        num_nodes = x.size(0)
        batch_size = num_nodes // self.max_length

        # Handle extra/missing nodes
        if num_nodes % self.max_length != 0:
            expected_nodes = batch_size * self.max_length
            if num_nodes > expected_nodes:
                x = x[:expected_nodes]
            else:
                pad_size = expected_nodes - num_nodes
                padding = torch.zeros((pad_size, x.size(1)), device=x.device)
                x = torch.cat([x, padding], dim=0)

        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = self.fc(x)

        x = x.view(batch_size, self.max_length, -1)
        return x

# Initialize Model, Loss, and Optimizer
input_dim = x.shape[1]
hidden_dim = 8
output_dim = len(label_map)

model = GCN(input_dim=input_dim, hidden_dim=hidden_dim, output_dim=output_dim, max_length=max_length)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)



Building graph: 100%|██████████| 15559/15559 [55:55<00:00,  4.64it/s] 


Data(x=[15559, 7527], edge_index=[2, 1985010], edge_weight=[1985010])


In [10]:
# Train Function
def train(model, data, optimizer, criterion, epochs=100, pad_value=0):
    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        out = model(data)

        out = out.view(-1, out.size(-1))  # Flatten predictions
        target = data.y.view(-1)  # Flatten targets

        mask = target != pad_value  # Ignore padding
        loss = criterion(out[mask], target[mask])
        loss.backward()
        optimizer.step()
        print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")

# Evaluate Function
def evaluate(model, data, pad_value=0):
    model.eval()
    with torch.no_grad():
        out = model(data)
        pred = out.argmax(dim=-1).view(-1)
        target = data.y.view(-1)

        mask = target != pad_value
        pred = pred[mask]
        target = target[mask]

        acc = accuracy_score(target.cpu(), pred.cpu())
        print(f"Accuracy: {acc:.4f}")
        print("\nClassification Report:")
        print(classification_report(target.cpu(), pred.cpu(), target_names=list(label_map.keys())))

# Train and Evaluate
train(model, gnn_data, optimizer, criterion, epochs=20)
evaluate(model, gnn_data)


Label tensor shape: torch.Size([15559, 32]), Label map: {'B': 0, 'C': 1, 'E': 2, 'G': 3, 'H': 4, 'I': 5, 'S': 6, 'T': 7}
